# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **ਸਾਫ਼ ਏਜੰਟ ਹਦਾਇਤਾਂ** — ਸਟੀਕ, ਭੂਮੀਕਾ-ਪਰਿਭਾਸ਼ਿਤ ਪ੍ਰਾਂਪਟ ਬਣਾਉਣਾ ਜੋ ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਦਾ ਮਾਰਗਦਰਸ਼ਨ ਕਰਦੇ ਹਨ  
2. **Pydantic ਮਾਡਲਾਂ ਨਾਲ ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** — ਇਹ ਸੁਨਿਸ਼ਚਿਤ ਕਰਨਾ ਕਿ ਏਜੰਟ ਅਣੁਮਾਨਯੋਗ, ਮੰਨਤਾ ਪ੍ਰਾਪਤ ਡੇਟਾ ਵਾਪਸ ਕਰਦੇ ਹਨ  
3. **ਇੱਕੱਲਾ ਜ਼ਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ** — ਕੇਂਦਰਿਤ ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਕਰਨਾ ਜੋ ਹਰ ਇੱਕ ਚੀਜ਼ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕਰਦੇ ਹਨ  

ਅਸੀਂ ਹਰ ਇੱਕ ਪੈਟਰਨ ਨੂੰ ਇੱਕ **ਯਾਤਰਾ ਗੰਤੀ ਸਿਫਾਰਸ਼ਕਾਰ** სცenario ਵਿੱਚ ਲਾਗੂ ਕਰਾਂਗੇ, ਲਗਾਤਾਰ ਇੱਕ ਐਸੀ ਪ੍ਰਣਾਲੀ ਤਿਆਰ ਕਰਦੇ ਹੋਏ ਜੋ ਸਥਾਨਾਂ ਦੀ ਸਿਫਾਰਸ਼ ਕਰ ਸਕੇ, ਉਪਲਬਧਤਾ ਜਾਂਚ ਸਕੇ, ਅਤੇ ਲੋਜਿਸਟਿਕਸ ਨੂੰ ਸੰਭਾਲ ਸਕੇ।


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ਅਦਾਗ਼ਾ 1: ਸਾਫ ਸੂਚਨਾ ਦੇਣਾ

ਸਭ ਤੋਂ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਅਦਾਗ਼ਾ ਸਭ ਤੋਂ ਸਧਾਰਣ ਵੀ ਹੈ: ਆਪਣੇ ਏਜੰਟ ਲਈ ਸਾਫ਼, ਵਿਸਥਾਰਪੂਰਕ ਹੁਕਮ ਲਿਖਣਾ।

ਚੰਗੀਆਂ ਸੂਚਨਾਵਾਂ ਤੈਅ ਕਰਦੀਆਂ ਹਨ:
- **ਕੌਣ** ਏਜੰਟ ਹੈ (ਪਿਛੋਕੜ ਅਤੇ ਅੰਦਾਜ਼)
- **ਕੀ** ਉਹ ਕਰੇ (ਕਦਮ-ਬੰਦ ਜ਼ਿੰਮੇਵਾਰੀਆਂ)
- **ਕਿਵੇਂ** ਉਹ ਵਰਤਾਉ ਕਰੇ (ਪਾਬੰਦੀਆਂ ਅਤੇ ਅੰਦਾਜ਼)

ਹੇਠਾਂ, ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਸਹਾਇਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸ ਨੂੰ ਖੁਲੇਤਰ੍ਹਾਂ ਸੂਚਨਾਵਾਂ ਦਿੱਤੀਆਂ ਗਈਆਂ ਹਨ ਜੋ ਉਸ ਦੀ ਹਰ ਜਵਾਬ ਨੂੰ ਸੰਵਾਰਦੀਆਂ ਹਨ।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic ਮਾਡਲਾਂ ਨਾਲ ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ

ਮੁਫ਼ਤ-ਰੂਪ ਪਾਠ ਗੱਲਬਾਤ ਲਈ ਲਾਭਦਾਇਕ ਹੁੰਦਾ ਹੈ, ਪਰ ਨੀਚਲੇ ਸਿਸਟਮਾਂ ਨੂੰ ਸੰਰਚਿਤ ਡਾਟਾ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।  
**Pydantic ਮਾਡਲਾਂ** ਨੂੰ ਇੱਕ **ਟੂਲ ਫੰਕਸ਼ਨ** ਨਾਲ ਜੋੜ ਕੇ, ਅਸੀਂ ਕਰ ਸਕਦੇ ਹਾਂ:

- ਏਜੰਟ ਦੇ ਆਉਟਪੁੱਟ ਲਈ ਇਕ ਵਿਸ਼ੇਸ਼ ਸਕੀਮਾ ਨਿਰਧਾਰਿਤ ਕਰਨਾ  
- ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਆਪ ਬੈਲਿਡੇਟ ਕਰਨਾ  
- ਏਜੰਟ ਦੇ ਨਤੀਜਿਆਂ ਨੂੰ ਐਪਲੀਕੇਸ਼ਨ ਲਾਜ਼ਿਕ ਵਿੱਚ ਭਰੋਸੇਯੋਗ ਢੰਗ ਨਾਲ ਸ਼ਾਮਲ ਕਰਨਾ  

ਅਸੀਂ ਇੱਕ ਐਸਾ ਟੂਲ ਵੀ ਪੇਸ਼ ਕਰਦੇ ਹਾਂ ਜੋ ਡੈਸਟਿਨੇਸ਼ਨ ਵੇਰਵੇ ਵਾਪਸ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਏਜੰਟ ਆਪਣੀਆਂ ਸਿਫਾਰਸ਼ਾਂ ਨੂੰ ਅਸਲੀ ਡਾਟਾ 'ਤੇ ਅਧਾਰਿਤ ਕਰ ਸਕੇ।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: ਸਿੰਗਲ ਰਿਸਪੌਂਸਬਿਲਿਟੀ ਏਜੰਟਸ

ਜਟਿਲ ਕੰਮਾਂ ਦਾ ਲਾਭ ਲੈਣ ਲਈ ਕਈ ਧਿਆਨ ਕੇਂਦਰਿਤ ਏਜੰਟਸ ਵਿੱਚ ਕੰਮ ਵੰਡਿਆ ਜਾਂਦਾ ਹੈ, ਹਰ ਇੱਕ ਦੀ ਇੱਕ ਹੀ ਜ਼ਿੰਮੇਵਾਰੀ ਹੁੰਦੀ ਹੈ:

- ਇੱਕ **ਡੈਸਟੀਨੇਸ਼ਨ ਐਕਸਪ੍ਰਟ** ਜੋ ਥਾਵਾਂ ਅਤੇ ਉਪਲੱਬਧਤਾ ਬਾਰੇ ਜਾਣਕਾਰ ਹੈ
- ਇੱਕ **ਲੌਜਿਸਟਿਕਸ ਪਲੈਨਰ** ਜੋ ਫਲਾਈਟਾਂ, ਹੋਟਲਾਂ, ਅਤੇ ਯਾਤ੍ਰਾ ਦਿਸ਼ਾ-ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸੰਭਾਲਦਾ ਹੈ

ਇਹ ਸੌਫਟਵੇਅਰ ਇੰਜੀਨੀਅਰਿੰਗ ਦੇ ਸਿਧਾਂਤ *ਤਫ਼ਰੀਕ-ਏ-ਮਸਾਇਲ* ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ — ਹਰ ਏਜੰਟ ਨੂੰ ਅਜ਼ਾਦੀ ਨਾਲ ਟੈਸਟ, ਸੰਭਾਲ ਅਤੇ ਸੁਧਾਰਣਾ ਅਸਾਨ ਹੁੰਦਾ ਹੈ।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ ਯਾਤਰਾ ਸਿਫਾਰਸ਼ੀ ਪਰਿਸਥਿਤੀ ਲਈ ਤਿੰਨ ਏਜੰਟਿਕ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਲਾਗੂ ਕੀਤੇ:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **ਸਾਫ ਨਿਰਦੇਸ਼** | ਪਹਿਚਾਣ, ਜ਼ਿੰਮੇਵਾਰੀਆਂ, ਅਤੇ ਸੀਮਾਵਾਂ ਪਹਿਲਾਂ ਤੋਂ ਪਰਿਭਾਸ਼ਿਤ ਕਰੋ | ਸਥਿਰ, ਬ੍ਰਾਂਡ ਅਨੁਕੂਲ ਏਜੰਟ ਵਿਵਹਾਰ |
| **ਸੰਰਚਿਤ ਨਤੀਜਾ** | ਜਵਾਬ ਫਾਰਮੈਟ ਵਜੋਂ Pydantic ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰੋ | ਵੈਧ, ਮਸ਼ੀਨ-ਪੜ੍ਹਨ ਯੋਗ ਨਤੀਜੇ |
| **ਇੱਕਲ ਪ੍ਰਤਿਬੱਧਤਾ** | ਹਰ ਏਜੰਟ ਨੂੰ ਇੱਕ ਕੇਂਦ੍ਰਿਤ ਕੰਮ ਦਿਓ | ਆਸਾਨ ਟੈਸਟਿੰਗ, ਰੱਖਰਖਾਵ, ਅਤੇ ਸੰਯੋਜਨ |

ਇਹ ਪੈਟਰਨ ਸਵਭਾਵਿਕ ਤੌਰ 'ਤੇ ਮਿਲਦੇ ਹਨ — ਤੁਸੀਂ ਇੱਕ-ਜ਼ਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ ਵਿੱਚ ਸਾਫ ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸੰਰਚਿਤ ਨਤੀਜੇ ਨਾਲ ਜੋੜ ਕੇ ਮਜਬੂਤ, ਉਤਪਾਦਨ-ਤੈਅਰ ਪ੍ਰਣਾਲੀਆਂ ਬਣਾ ਸਕਦੇ ਹੋ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸੱਤੀਕਰਨ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਹੀਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਅਸਲ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਇਸਦੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮੁੱਖ ਸਰੋਤ ਮੰਨਿਆ ਜਾਵੇ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪ੍ਰੋਫੈਸ਼ਨਲ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਉਤਪੰਨ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਅਸੀਂ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹੋਵਾਂਗੇ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
